# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a practical guide for loading, exploring, and processing the FAIR² colorectal cancer dataset using the [mlcroissant](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is defined by a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Install mlcroissant if not already installed
!pip install -U mlcroissant

## 1. Data Loading

Load the dataset metadata and prepare for exploration.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Version: {metadata.version}")
print(f"Citation: {metadata.citeAs}")
print(f"Published: {metadata.datePublished}")


## 2. Data Overview

Review available record sets and fields. All are referenced by their `@id` values as per the Croissant standard.

In [ ]:
# List all available record sets in the dataset with their @id and label (if any)
print("Record sets defined in the dataset:")

record_sets = metadata.recordSets
if not record_sets:
    print('No record sets found in the metadata!')
else:
    for rs in record_sets:
        print(f"@id: {rs['@id']} | Name: {rs.get('name', '(no name)')}")

# If no recordSets in metadata API, fallback to auto-discover using Dataset API
if not record_sets:
    print("\nAttempting to discover record sets with mlcroissant.Dataset API...")
    discovered_ids = set()
    for rs_meta in dataset.record_sets():
        rs_id = rs_meta['@id']
        discovered_ids.add(rs_id)
        print(f"@id: {rs_id} | Name: {rs_meta.get('name', '(no name)')}")
else:
    discovered_ids = {rs['@id'] for rs in record_sets}


Inspect the fields and columns of the first record set, referencing all components by their `@id`.

In [ ]:
# For demonstration, choose the first found record set
if discovered_ids:
    selected_record_set_id = list(discovered_ids)[0]
    print(f'Selected Record Set for deeper inspection: {selected_record_set_id}')

    # Display a sample record and all field @id's and column @id's
    print('\nSample record from the selected record set:')
    sample_record = next(dataset.records(record_set=selected_record_set_id), None)
    if sample_record:
        pprint.pprint(sample_record)
        print('\nField @id values in this record set:')
        for field in dataset.fields(record_set=selected_record_set_id):
            print(f"  Field @id: {field['@id']} | Name: {field.get('name', '(no name)')}")

        print('\nColumn @id values and parent fields:')
        for field in dataset.fields(record_set=selected_record_set_id):
            if 'column' in field:
                for column in (field['column'] if isinstance(field['column'], list) else [field['column']]):
                    print(f"  Column @id: {column['@id']} (Field: {field['@id']})")
    else:
        print('No records found in this record set.')
else:
    print('No record set @id available for inspection.')


## 3. Data Extraction

Load all records from the main record set into a DataFrame for analysis. Use `@id` references.

In [ ]:
# Gather all record set @id values for bulk loading
record_sets_ids = list(discovered_ids)
dataframes = {}

for rs_id in record_sets_ids:
    # Each record is a dict; keys are field @id or names, values are values
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records from record set '@id': {rs_id}")
        print(f"Columns (fields referenced by @id): \n{df.columns.tolist()}\n")
    else:
        print(f"Record set '@id' {rs_id} had no records loaded.")

# For further examples, pick the first non-empty DataFrame
main_record_set_id = None
for rs_id, df in dataframes.items():
    if not df.empty:
        main_record_set_id = rs_id
        break

if main_record_set_id:
    print(f"Example preview from record set '@id': {main_record_set_id}")
    display(dataframes[main_record_set_id].head())
else:
    print('No populated record set found for demonstration.')


## 4. Exploratory Data Analysis (EDA)

Let's filter, normalize, and group using the actual field `@id`s in the loaded DataFrame. Identify a numeric field and a grouping field using the metadata above.

In [ ]:
# For demonstration: identify numeric and groupable fields
if main_record_set_id:
    df = dataframes[main_record_set_id]

    # Try to automatically guess a numeric field and group field using data types
    numeric_candidates = df.select_dtypes(include='number').columns.tolist()
    if not numeric_candidates:
        # Try converting columns that look like age or similar
        for col in df.columns:
            if 'age' in col.lower():
                df[col] = pd.to_numeric(df[col], errors='coerce')
        numeric_candidates = df.select_dtypes(include='number').columns.tolist()
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
    else:
        numeric_field_id = df.columns[0] # fallback

    # For group field, prefer a categorical
    group_candidates = [col for col in df.columns if col != numeric_field_id and (df[col].dtype == object or str(df[col].dtype)=='category')]
    group_field_id = group_candidates[0] if group_candidates else None

    print(f"Using numeric field @id: {numeric_field_id}")
    if group_field_id:
        print(f"Using group field @id: {group_field_id}")
    
    # Filter: show records with numeric_field > threshold (pick mean or quantile for test)
    threshold = df[numeric_field_id].quantile(0.8) if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else None

    if threshold is not None:
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize numeric field
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Grouping if group_field exists
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
            display(grouped_df.head())
    else:
        print(f"No numeric field detected for filtering in the sample data. Columns: {df.columns.tolist()}")
else:
    print('Main record set DataFrame not available for EDA.')


## 5. Visualization

Visualize a numerical field's distribution and, if possible, the relationship to a group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_id in dataframes[main_record_set_id].columns:
    plt.figure(figsize=(8,4))
    sns.histplot(data=dataframes[main_record_set_id], x=numeric_field_id, bins=15, kde=True)
    plt.title(f"Distribution of Numeric Field '@id': {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id and group_field_id in dataframes[main_record_set_id].columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=dataframes[main_record_set_id])
        plt.title(f"{numeric_field_id} by '{group_field_id}'")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print('Cannot create visualization: numeric or group field missing in loaded DataFrame.')


## 6. Conclusion

In this notebook, we demonstrated how to load and explore the FAIR² colorectal cancer dataset using `mlcroissant`. Using Croissant `@id` references ensures machine-actionable and reproducible access to biomedical data schemas. Users can extend this approach for advanced modeling or domain-specific analyses using the same consistent API.